# Дубли `ods_alpha.scd1_trx_int` — март 2026

Отдельная проверка для озера. **Не** смешивать с майским Excel vs `final_df`.

Тот же SQL, что секция **5f** в `excel_march_2026_chod_finrez_dip.ipynb` и файл `lake_march_2026_trx_int_dups.sql`.

Периметр витрины, один месяц, без скана всей `trx_int`:
1. `scd1_base24_fiids` — RSHB
2. `scd1_agreements` — SA, живые в месяце
3. `scd1_trx` — SA/S01, не reversed, `d_trx_orig` в месяце
4. `scd1_trx_acq` — `n_agr` из SA
5. живые `scd1_trx_int` только по этим `n_trx`

`trx_int` даты транзакции не имеет — месяц берётся из `scd1_trx.d_trx_orig`.

**Ожидание (уже снимали):**

| месяц | `multi_same_fee_dup` | `avg_rows_on_multi` | `multi_different_fees` | `extra_fee_from_same_fee_dups` |
|---|---|---|---|---|
| 2026-03 | ≈ **666 739** | ≈ **2.00** | **0** | ≈ **−5.97e6** |
| 2026-02 | ≈ **1** | — | — | — |
| 2026-04 | ≈ **0** | — | — | — |

Запускать ячейки сверху вниз. При Kerberos `500168` — эта тетрадка сама переподключается (`FORCE_RECONNECT=True`).


In [ ]:
from calendar import monthrange

from IPython.display import display
import pandas as pd
from rail_connectors.connection import connect

MEM_LIMIT = '16g'
FORCE_RECONNECT = True

if FORCE_RECONNECT:
    imp = None

if 'imp' not in globals() or imp is None:
    imp = connect(
        to='IMPALA',
        extra_options={'db': 'sandbox_ai'},
        driver_args={'tez.queue.name': 'ai'},
        kerberos={
            'keytab_path': '/home/jovyan/test_requests/tech.keytab',
            'use_credentials': True,
            'update_keytab': True,
        },
        user_params={'user_name': 'Shestopalov-VYur'},
    )
    imp._init_connection()
    print('Impala connected')
else:
    print('Reuse existing imp')


def run_sql(sql, title=None):
    if title:
        print(title)
    with imp:
        imp.execute(f'set MEM_LIMIT={MEM_LIMIT}')
        df = imp.fetch(sql)
    display(df)
    return df


## SQL

Не запускать `GROUP BY n_trx` по всей `scd1_trx_int` без ключей месяца — упадёт по памяти.


In [ ]:
def month_bounds(ym):
    y, m = map(int, ym.split('-'))
    return f'{ym}-01', f'{ym}-{monthrange(y, m)[1]:02d}'


def cte_trx_keys(ms, me):
    return f"""
    fiid_rshb as (
      select distinct cast(fa.c_fiid as string) as c_fiid
      from ods_alpha.scd1_base24_fiids fa
      where coalesce(cast(fa.c_fiid_grp as string), 'UNKNOWN') = 'RSHB'
    ),
    sa_agr as (
      select distinct cast(a.n_agr as string) as n_agr
      from ods_alpha.scd1_agreements a
      where upper(trim(cast(a.acq_class as string))) = 'SA'
        and cast(a.d_valid_from as date) <= cast('{me}' as date)
        and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{ms}' as date))
        and coalesce(a.ods_deleted_flg, '0') <> '1'
    ),
    trx_base as (
      select cast(t.n_trx as string) as n_trx
      from ods_alpha.scd1_trx t
      join fiid_rshb fr on fr.c_fiid = cast(t.c_fiid_acq as string)
      where cast(t.d_trx_orig as timestamp) >= cast('{ms}' as timestamp)
        and cast(t.d_trx_orig as timestamp) < cast(date_add(cast('{me}' as date), 1) as timestamp)
        and t.c_nter is not null
        and coalesce(t.ods_deleted_flg, '0') <> '1'
        and t.c_trx_class = 'SA'
        and t.c_trx_type = 'S01'
        and coalesce(t.cf_trx_stat, '') <> 'R'
      group by cast(t.n_trx as string)
    ),
    ta as (
      select cast(a.n_trx as string) as n_trx
      from ods_alpha.scd1_trx_acq a
      join trx_base tb on tb.n_trx = cast(a.n_trx as string)
      join sa_agr ss on ss.n_agr = cast(a.n_agr as string)
      where coalesce(a.ods_deleted_flg, '0') <> '1'
      group by cast(a.n_trx as string)
    )
    """


def sql_multi_class(ym):
    ms, me = month_bounds(ym)
    return f"""
    with {cte_trx_keys(ms, me)},
    int_alive as (
      select
        cast(i.n_trx as string) as n_trx,
        coalesce(cast(i.n_amt_fee as double), 0.0) as n_amt_fee
      from ods_alpha.scd1_trx_int i
      join ta k on k.n_trx = cast(i.n_trx as string)
      where coalesce(i.ods_deleted_flg, '0') <> '1'
    ),
    per_trx as (
      select
        n_trx,
        count(*) as int_rows,
        count(distinct cast(n_amt_fee as string)) as distinct_fee_values,
        sum(n_amt_fee) as fee_sum,
        max(n_amt_fee) as fee_max
      from int_alive
      group by n_trx
    ),
    multi as (
      select * from per_trx where int_rows > 1
    )
    select
      '{ym}' as report_month,
      count(*) as multi_trx_cnt,
      sum(case when distinct_fee_values = 1 then 1 else 0 end) as multi_same_fee_dup,
      sum(case when distinct_fee_values > 1 then 1 else 0 end) as multi_different_fees,
      avg(int_rows) as avg_rows_on_multi,
      sum(case when distinct_fee_values = 1 then fee_sum - fee_max else 0 end) as extra_fee_from_same_fee_dups
    from multi
    """


print('SQL helpers ready')
print(sql_multi_class('2026-03')[:400], '...')


## Прогон: февраль / март / апрель

Один месяц за раз. Не параллелить с другими тяжёлыми Impala-запросами.


In [ ]:
EXPECTED = {
    '2026-02': {'multi_same_fee_dup': (0, 5)},
    '2026-03': {'multi_same_fee_dup': (650_000, 680_000), 'avg_rows': (1.95, 2.05), 'diff_fees': (0, 0)},
    '2026-04': {'multi_same_fee_dup': (0, 5)},
}

DUP_MONTHS = ['2026-02', '2026-03', '2026-04']
parts = []
for ym in DUP_MONTHS:
    part = run_sql(sql_multi_class(ym), f'=== дубли trx_int {ym} ===')
    if part is not None and len(part):
        parts.append(part)

dup_by_month = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
print('\n=== свод ===')
display(dup_by_month)

print('\n=== сверка с эталоном ===')
ok_all = True
for _, r in dup_by_month.iterrows():
    ym = str(r['report_month'])
    same = int(r['multi_same_fee_dup'] or 0)
    exp = EXPECTED[ym]
    lo, hi = exp['multi_same_fee_dup']
    ok = lo <= same <= hi
    if ym == '2026-03':
        avg = float(r['avg_rows_on_multi'] or 0)
        diff = int(r['multi_different_fees'] or 0)
        ok = ok and (exp['avg_rows'][0] <= avg <= exp['avg_rows'][1]) and diff == 0
        print(
            f"{ym}: same_fee_dup={same:,}  avg_rows={avg:.4f}  different_fees={diff}  "
            f"extra_fee={r['extra_fee_from_same_fee_dups']}  → {'OK' if ok else 'FAIL'}"
        )
    else:
        print(f'{ym}: same_fee_dup={same:,}  → {"OK" if ok else "FAIL"}  (ожидали {lo}…{hi})')
    ok_all = ok_all and ok

print('\nИТОГ:', 'SQL отрабатывает, цифры как раньше' if ok_all else 'цифры не совпали с эталоном — смотри свод')


## Sample: две живые строки на один мартовский `n_trx`

Ожидание: одинаковый `n_amt_fee`, `ods_deleted_flg=0`, часто `ods_op_type='SQL COMPUPDATE'`.


In [ ]:
ms, me = month_bounds('2026-03')
sql_sample = f"""
with {cte_trx_keys(ms, me)},
multi as (
  select cast(i.n_trx as string) as n_trx
  from ods_alpha.scd1_trx_int i
  join ta k on k.n_trx = cast(i.n_trx as string)
  where coalesce(i.ods_deleted_flg, '0') <> '1'
  group by cast(i.n_trx as string)
  having count(*) > 1
     and count(distinct cast(i.n_amt_fee as string)) = 1
  limit 8
)
select
  cast(i.n_trx as string) as n_trx,
  cast(i.n_amt_fee as double) as n_amt_fee,
  cast(i.ods_deleted_flg as string) as ods_deleted_flg,
  cast(i.ods_op_type as string) as ods_op_type
from ods_alpha.scd1_trx_int i
join multi m on m.n_trx = cast(i.n_trx as string)
where coalesce(i.ods_deleted_flg, '0') <> '1'
order by 1, 2
"""

sample = run_sql(sql_sample, '=== sample дублей март ===')
if sample is not None and len(sample):
    per = sample.groupby('n_trx').size()
    print('строк на n_trx:', per.to_dict())
    print('уникальных fee на n_trx:', sample.groupby('n_trx')['n_amt_fee'].nunique().to_dict())
